In [ ]:
import torch
from torch import nn
from d2l import torch as d2l
def corr2d(x,k):
    h,w=k.shape
    y=torch.zeros((x.shape[0]-h+1),x.shape[1]-w+1)# 输出高度 = 输入高度 - 核高度 + 1，# 输出宽度 = 输入宽度 - 核宽度 + 1
    for i in range(y.shape[0]):
        for j in range(y.shape[1]):
            y[i,j]=(x[i:i+h,j:j+w]*k).sum()
    return y


In [3]:
X = torch.tensor([[0.0, 1.0, 2.0],
                  [3.0, 4.0, 5.0],
                  [6.0, 7.0, 8.0]])

K = torch.tensor([[0.0, 1.0],
                  [2.0, 3.0]])
corr2d(X,K)


tensor([[19., 25.],
        [37., 43.]])

In [ ]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))#parameter，自动学习和更新的变量，主要包括权重，torch.rand(kernel_size)，随机初始化权重矩阵
        self.bias = nn.Parameter(torch.zeros(1))#偏置初始化为0
    
    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

In [5]:
x=torch.ones((6,8))
x[:,2:6]=0
x

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [6]:
k=torch.tensor([[1.0,-1.0]])

In [8]:
y=corr2d(x,k)
y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [9]:
#卷积核k只可以检测垂直边缘
corr2d(x.t(),k)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

In [11]:
conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)

x = x.reshape((1, 1, 6, 8))
y = y.reshape((1, 1, 6, 7))

for i in range(10):
    y_hat = conv2d(x)
    l = (y_hat - y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data[:] -= 3e-2 * conv2d.weight.grad
    
    if (i + 1) % 2 == 0:
        print(f'batch {i+1}, loss {l.sum():.3f}')

batch 2, loss 10.416
batch 4, loss 2.127
batch 6, loss 0.513
batch 8, loss 0.150
batch 10, loss 0.051
